# This Notebook is used to create a visual representation of the Localization offset 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import PIL
import json
from PIL import Image
import os

In [ ]:
loc_poses=[]
loc_vertex_ids = []
loc_vertex_times = []
times = []
folder_path = '/home/adam/Desktop/CurrentBranch/src/main/src/vtr_db_extractor/loc_results'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files: {file_count}")
for i in range(0,file_count,1):
    filename = filename=folder_path+f"/T_r_vertex_{i:04d}.txt"
    matrix = []
    
    with open(filename, 'r') as file:
        for line in file:
            # Skip empty lines
            if line.startswith('vertex_id:'):
                vertex_id =int(line.split(": ")[1].strip())
                loc_vertex_ids.append(vertex_id)
            elif line.startswith('vertex_timestamp:'):
                vertex_time =int(line.split(": ")[1].strip())
                loc_vertex_times.append(vertex_time)
            elif line.startswith("timestamp:"):
                time =int(line.split(": ")[1].strip())
                times.append(time)
            elif line.strip() and not line.startswith('u'):
                # Split the line by whitespace and convert each element to float
                row = [float(val) for val in line.strip().split()]
                matrix.append(row)
            

    temp = np.array(matrix[0:4])
    loc_poses.append(np.array(matrix[0:4]))
poses=np.array(loc_poses)
times= np.array(times)
loc_vertex_ids = np.array(loc_vertex_ids)
loc_vertex_times= np.array(loc_vertex_times)

In [ ]:
graph_poses=[]
graph_from_ids=[]
graph_to_ids=[]
graph_edge_types=[]
folder_path = '/home/adam/Desktop/CurrentBranch/src/main/src/vtr_db_extractor/edge_results'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
file_count =int((file_count-1))
print(f"Number of files: {file_count}")
for i in range(0,file_count,1):
    filename = filename=folder_path+f"/edge_{i:04d}.txt"
    matrix = []
    with open(filename, 'r') as file:
        for line in file:
            # Skip empty lines
            if line.startswith('from_vertex_id'):
                # Extract the vertex ID from the line
                from_vertex_id = int(line.split(':')[1].strip())
                # print(f"From Vertex ID: {from_vertex_id}")
                graph_from_ids.append(from_vertex_id)
            elif line.startswith('to_vertex_id'):
                # Extract the vertex ID from the line
                to_vertex_id = int(line.split(':')[1].strip())
                # print(f"To Vertex ID: {to_vertex_id}")
                graph_to_ids.append(to_vertex_id)
            elif line.startswith('edge_type'):
                # Extract the vertex ID from the line
                edge_type = int(line.split(':')[1].strip())
                # print(f"To Vertex ID: {to_vertex_id}")
                graph_edge_types.append(edge_type)
            elif line.strip() and not line.startswith('u'):
                # Split the line by whitespace and convert each element to float
                row = [float(val) for val in line.strip().split()]
                matrix.append(row)

    temp = np.array(matrix[0:4])
    graph_poses.append(np.array(matrix[0:4]))
# poses=np.array(graph_poses)

# Sanity check 
poses=[]
from_ids=[]
to_ids=[]
edge_types=[]

for i in range(0,graph_from_ids.__len__(),1):
    if graph_edge_types[i] == 0 and graph_from_ids[i]<1000000:
        from_ids.append(graph_from_ids[i])
        to_ids.append(graph_to_ids[i])
        edge_types.append(graph_edge_types[i])
        poses.append(graph_poses[i])

def bubble_sort(arr1, arr2, arr3):
    for i in range(len(arr1)):
        # Last i elements are already in place
        for j in range(0, len(arr1) - i - 1):
            if arr1[j] > arr1[j + 1]:
                # Swap if elements are in the wrong order
                arr1[j], arr1[j + 1] = arr1[j + 1], arr1[j]
                arr2[j], arr2[j + 1] = arr2[j + 1], arr2[j]
                arr3[j], arr3[j + 1] = arr3[j + 1], arr3[j]
    return np.array(arr1), np.array(arr2), np.array(arr3)

sorted_from, sorted_to, sorted_poses = bubble_sort(from_ids.copy(), to_ids.copy(), poses.copy())
print("Sorted list:", sorted_from)
print("Corresponding to list:", sorted_to)

# creating new matrices:
global_poses = []
pose1 = np.array([
    [-1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])
global_poses.append(pose1)
for i in range(0,sorted_from.__len__(),1):
    temp_pose =  global_poses[i] @ sorted_poses[i] 
    global_poses.append(temp_pose)
print(global_poses[0])
global_poses = np.array(global_poses)

origins = global_poses[:, :3, -1]
print(origins[0:3])
dirs = np.stack([np.sum([1, 0, 0] * pose[:3, :3], axis=-1) for pose in global_poses])
x= origins[..., 0].flatten()
y= origins[..., 1].flatten()
z= origins[..., 2].flatten()
#print(np.array(x))
# print(type(x))

%matplotlib ipympl
s = np.ones(len(x))
#plt.scatter(x=x, y=y, color='red')
plt.scatter(x, y, s=s, color='midnightblue', label='Group 1')
# plt.xlim(0, 35)   
# plt.ylim(-7, 1) 
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Pose-Graph Edges")
plt.axis('equal')
plt.show()

In [ ]:
print((global_poses.shape))

loc_global_poses = []
for i in range(len(loc_poses)):
    # print(loc_vertex_ids[i])
    # print()
    temp_pose =  global_poses[int(loc_vertex_ids[i])] @ loc_poses[i] 
    loc_global_poses.append(temp_pose)
loc_global_poses = np.array(loc_global_poses)


In [ ]:
%matplotlib ipympl
loc_origins = loc_global_poses[:, :3, -1]
loc_dirs = np.stack([np.sum([1, 0, 0] * pose[:3, :3], axis=-1) for pose in loc_global_poses])
loc_x= loc_origins[..., 0].flatten()
loc_y= loc_origins[..., 1].flatten()
loc_z= loc_origins[..., 2].flatten()

graph_origins = global_poses[:, :3, -1]
graph_dirs = np.stack([np.sum([1, 0, 0] * pose[:3, :3], axis=-1) for pose in global_poses])
graph_x= graph_origins[..., 0].flatten()
graph_y= graph_origins[..., 1].flatten()
graph_z= graph_origins[..., 2].flatten()

#print(np.array(x))
# print(type(x))

# s_loc = np.ones(len(loc_x))
# s_graph = np.ones(len(graph_x))
plt.scatter(loc_x, loc_y, s=1, alpha=0.8, color='red', label='Localization Estimate')
plt.scatter(graph_x, graph_y, s=1, alpha=0.8, color='midnightblue',label='Graph location' )

# plt.xlim(0, 35)   
# plt.ylim(-7, 1) 
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.title("Localization result")
plt.axis('equal')
plt.show()